## Load cleaned data

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

data = pd.read_csv("outputs/cleaned_data.csv", parse_dates=["Order Date"])
data.shape

(1452, 9)

In [2]:
data.head()

,Order Date,Sales,year,month,day,day_of_week,quarter,sales_lag1,rolling_mean_7
0,2014-01-09,40.544,2014,1,9,3,1,0.000,694.120857
1,2014-01-10,54.830,2014,1,10,4,1,40.544,699.604000
2,2014-01-11,9.940,2014,1,11,5,1,54.830,659.872571
3,2014-01-12,0.000,2014,1,12,6,1,9.940,657.081714
4,2014-01-13,3553.795,2014,1,13,0,1,0.000,535.181000


## Train test split

In [3]:
features = ["year", "month", "day", "day_of_week", "quarter", "sales_lag1", "rolling_mean_7"]
target = "Sales"

split_index = int(len(data) * 0.8)
train = data.iloc[:split_index]
test = data.iloc[split_index:]

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

print(X_train.shape, X_test.shape)

(1161, 7) (291, 7)


## Linear regression baseline

In [4]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

In [5]:
lr_mae = mean_absolute_error(y_test, lr_preds)
lr_rmse = mean_squared_error(y_test, lr_preds) ** 0.5
print("linear regression")
print("MAE:", lr_mae)
print("RMSE:", lr_rmse)

linear regression
MAE: 1649.9155703855993
RMSE: 2265.7430175580553


## Random forest model

In [6]:
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

In [7]:
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = mean_squared_error(y_test, rf_preds) ** 0.5
print("random forest")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)

random forest
MAE: 1658.9240478917525
RMSE: 2333.872830740519


## Compare results

In [8]:
results = pd.DataFrame({
    "model": ["Linear Regression", "Random Forest"],
    "MAE": [lr_mae, rf_mae],
    "RMSE": [lr_rmse, rf_rmse]
})
results

,model,MAE,RMSE
0,Linear Regression,1649.915570,2265.743018
1,Random Forest,1658.924048,2333.872831


## Save predictions

In [9]:
output = test[["Order Date", "Sales"]].copy()
output["predicted_sales"] = rf_preds
output.to_csv("outputs/predictions.csv", index=False)